# Movie Recommendation System
This notebook demonstrates the implementation of a content-based movie recommendation system using the TMDB dataset, including movie poster integration.

## 1. Importing Libraries and Loading Data

In [1]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [2]:
# Load datasets
movies = pd.read_csv('Data/movies.csv')
credits = pd.read_csv('Data/credits.csv')
posters = pd.read_csv('Data/poster.csv')

print("Datasets loaded successfully.")

Datasets loaded successfully.


## 2. Data Cleaning and Merging
We will merge the `movies`, `credits`, and `poster` datasets on the movie title.

In [3]:
# Merge datasets on title
movies = movies.merge(credits, on='title')
movies = movies.merge(posters, on='title')

# Select relevant columns
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'poster']]

# Check for missing values
movies.dropna(inplace=True)
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,poster
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",https://image.tmdb.org/t/p/original/jRXYjXNq0C...
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",https://image.tmdb.org/t/p/original/2YMnBRh8F6...
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",https://image.tmdb.org/t/p/original/zj8ongFhtW...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",https://image.tmdb.org/t/p/original/85cWkCVfti...
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",https://image.tmdb.org/t/p/original/7GSSyUUgUE...


## 3. Feature Engineering
Extract relevant information from JSON-like structures in `genres`, `keywords`, `cast`, and `crew`.

In [4]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [5]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert3)

In [6]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [7]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

### Removing spaces from tags

In [8]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

## 4. Creating the Similarity Model

In [9]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new_df = movies[['movie_id', 'title', 'tags', 'poster']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())
new_df.head()

/var/folders/fy/sj43z68s65g67n7z6vwkdtrw0000gn/T/ipykernel_9879/998875263.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
/var/folders/fy/sj43z68s65g67n7z6vwkdtrw0000gn/T/ipykernel_9879/998875263.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


,movie_id,title,tags,poster
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...",https://image.tmdb.org/t/p/original/jRXYjXNq0C...
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...",https://image.tmdb.org/t/p/original/2YMnBRh8F6...
2,206647,Spectre,a cryptic message from bond’s past sends him o...,https://image.tmdb.org/t/p/original/zj8ongFhtW...
3,49026,The Dark Knight Rises,following the death of district attorney harve...,https://image.tmdb.org/t/p/original/85cWkCVfti...
4,49529,John Carter,"john carter is a war-weary, former military ca...",https://image.tmdb.org/t/p/original/7GSSyUUgUE...


In [10]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vectors)

## 5. Recommendation System in Action

In [11]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"Recommendations for '{movie}':")
    for i in movies_list:
        m = new_df.iloc[i[0]]
        print(f"- {m.title} (Poster: {m.poster})")

recommend('Avatar')

Recommendations for 'Avatar':
- Aliens (Poster: https://image.tmdb.org/t/p/original/r1x5JGpyqZU8PYhbs4UcrO1Xb6x.jpg)
- Titan A.E. (Poster: https://image.tmdb.org/t/p/original/el2iHk3LTJWfEnwrvcRkvWY501G.jpg)
- Lifeforce (Poster: https://image.tmdb.org/t/p/original/953hMDf9G2ZRIEs97M6iFIYWtWF.jpg)
- Independence Day (Poster: https://image.tmdb.org/t/p/original/p0BPQGSPoSa8Ml0DAf2mB2kCU0R.jpg)
- Small Soldiers (Poster: https://image.tmdb.org/t/p/original/2nuUjSzHsoYlRvTPmLo7m7gCQry.jpg)


## 6. Saving the Model

In [12]:
pickle.dump(new_df, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))